# 统计量 vs ASTRA 交叉验证演示

把测试层第 3 层 (test_cross_validation.py) 变成可视化: 我们的统计
与 ASTRA 自身 Xemit/Zemit 逐列对照。这是"数据正确性"的直接证据。

In [ ]:
%run ../notebooks/_bootstrap.py

In [ ]:
import numpy as np
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics

base = PROJECT_ROOT / "examples" / "Manual_Example"
dist = read_distribution(base / "Example.0150.001")
s = compute_statistics(dist)
xemit = np.loadtxt(base / "Example.Xemit.001")[-1]
zemit = np.loadtxt(base / "Example.Zemit.001")[-1]

rows = [
    ("sigma_x [mm]", s.sig_x * 1e3, xemit[3]),
    ("sigma_x' [mrad]", s.sig_xp * 1e3, xemit[4]),
    ("eps_nx [pi mm mrad]", s.emit_x_norm * 1e6, xemit[5]),
    ("mean E_kin [MeV]", s.mean_E_kin_eV * 1e-6, zemit[2]),
    ("sigma_z [mm]", s.sig_z * 1e3, zemit[3]),
    ("sigma_E [keV]", s.sig_E_eV * 1e-3, zemit[4]),
    ("eps_nz [keV mm]", s.sig_E_eV * 1e-3 * s.sig_z * 1e3, zemit[5]),
]
print("%-22s %-14s %-14s %s" % ("量", "我们", "ASTRA", "rel"))
for name, ours, ref in rows:
    rel = abs(ours - ref) / abs(ref) * 100
    flag = "OK" if rel < 0.5 else "MISMATCH"
    print("%-22s %-14.6g %-14.6g %.4f%% %s" % (name, ours, ref, rel, flag))

In [ ]:
# 螺线管正则动量: 发射度补偿算例必须用 canonical momentum
from scipy.interpolate import interp1d
table = np.loadtxt(base / "Solenoid.dat")
bz = float(interp1d(table[:, 0], table[:, 1])(1.5 - 1.2)
           * 0.35 / table[:, 1].max())
s_canon = compute_statistics(dist, bz_on_axis_T=bz)
print("Bz(束心) = %.4f T" % bz)
print("eps_nx (canonical): %.4f pi mm mrad (Xemit: %.4f)" % (
    s_canon.emit_x_norm * 1e6, xemit[5]))
print("eps_nx (裸 px):     %.4f pi mm mrad" % (s.emit_x_norm * 1e6))

要点: 本算例束团位于螺线管尾场中, 只有用正则动量
px + c*Bz*y/2 计算的发射度才与 ASTRA 一致 (物理规则 4)。